# MODULES

In [1]:
!pip install pathlib
!pip install matplotlib

In [2]:
!nvidia-smi

Sat Oct 11 20:04:57 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3060        Off | 00000000:01:00.0  On |                  N/A |
|  0%   44C    P8              13W / 170W |   1613MiB / 12288MiB |     28%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

1.13.1+cu117
11.7
NVIDIA GeForce RTX 3060


In [4]:
import cv2
import os
import math
import time
import random
import pathlib
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# GLOBAL VARIABLES

In [5]:
ROOT_DIR_IMAGES = '../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation'

# IMAGES

In [6]:
def get_image_paths(root_dir, include='*', exclude=[]):
  paths = []
  
  for path in pathlib.Path(root_dir).glob(include): 
    paths.append(path)

  paths = sorted(paths)

  return paths

    
def base_filename_organization(imagPaths):
    baseFileNames = {}
    for imagPath in imagPaths:
        baseFileName = '_'.join(str(imagPath.stem).split('_')[:-1])
        if baseFileName not in baseFileNames:
            baseFileNames[baseFileName] = []
        baseFileNames[baseFileName].append(imagPath)

    # Sort lists numerically by the last suffix
    for key in baseFileNames:
        baseFileNames[key].sort(
            key=lambda p: int(p.stem.split('_')[-1])
        )

    return baseFileNames


def remove_last_element(baseFileNames):
    for baseFileName in baseFileNames:
        imagPaths = baseFileNames[baseFileName]
        length = len(imagPaths)
        baseFileNameLength = baseFileName + f'_{length-1}'
        imagPaths = [imagPath for imagPath in imagPaths if baseFileNameLength not in str(imagPath)]
        baseFileNames[baseFileName] = imagPaths
    return baseFileNames


def rotate_image(image, angle):
    """Rotate image around its center without cropping."""
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    
    # compute new bounding dimensions
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))
    
    # adjust rotation matrix
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    
    return cv2.warpAffine(image, M, (new_w, new_h))

def make_composite_images(images_dict: dict, final_width: int, final_height: int, save_dir: str):
    """
    Creates composite images from dict of images using OpenCV.
    Generates 6 versions for each key, each with a global rotation applied.
    
    Args:
        images_dict (dict): {key: [list of image paths]}
        final_width (int): width of final composite image
        final_height (int): height of final composite image
        save_dir (str): directory to save composite results
    """
    pathlib.Path(save_dir).mkdir(parents=True, exist_ok=True)

    # Rotations to apply
    rotations = [-30, -15, 0, 15, 30]

    for key, img_paths in images_dict.items():
        n = len(img_paths)
        grid_size = math.ceil(math.sqrt(n))  # square-ish grid
        cell_w = final_width // grid_size
        cell_h = final_height // grid_size

        # Load all images once
        loaded_images = []
        for img_path in img_paths:
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"⚠️ Could not load {img_path}")
                continue
            loaded_images.append(img)

        for i, angle in enumerate(rotations):
            # Black canvas
            composite = np.zeros((final_height, final_width, 3), dtype=np.uint8)

            for idx, img in enumerate(loaded_images):
                try:
                    rotated = rotate_image(img, angle)
                    resized = cv2.resize(rotated, (cell_w, cell_h), interpolation=cv2.INTER_AREA)

                    row, col = divmod(idx, grid_size)
                    y, x = row * cell_h, col * cell_w
                    composite[y:y+cell_h, x:x+cell_w] = resized
                except Exception as e:
                    print(f"⚠️ Error with image idx {idx}: {e}")

            out_path = str(pathlib.Path(save_dir) / f"{key}_{i+1}.jpg")
            cv2.imwrite(out_path, composite)
            print(f"✅ Saved {out_path} (rotation {angle}°)")


In [10]:
imagsPaths = {}
imagsPaths['fall']    = get_image_paths( ROOT_DIR_IMAGES + '/Fall/Raw_Image')
imagsPaths['no_fall'] = get_image_paths( ROOT_DIR_IMAGES + '/No_Fall/Raw_Image')

print(f"[INFO] videos dict keys: {list(imagsPaths.keys())}")
print(f"[INFO] videos['fall'] has {len(imagsPaths['fall'])} entries")
print(f"[INFO] videos['no_fall'] has {len(imagsPaths['no_fall'])} entries")

[INFO] videos dict keys: ['fall', 'no_fall']
[INFO] videos['fall'] has 400 entries
[INFO] videos['no_fall'] has 368 entries


In [11]:
BASEFILENAMES = {}

BASEFILENAMES['fall'] = base_filename_organization(imagsPaths['fall'])
BASEFILENAMES['no_fall'] = base_filename_organization(imagsPaths['no_fall'])
print( f"Fall {len(BASEFILENAMES['fall'])}")
print( f"No fall {len(BASEFILENAMES['no_fall'])}")


Fall 25
No fall 23


## Remove last image

In [15]:
BASEFILENAMES['fall'] = remove_last_element(BASEFILENAMES['fall'])
BASEFILENAMES['no_fall'] = remove_last_element(BASEFILENAMES['no_fall'])
print( f"Fall {len(BASEFILENAMES['fall'])}")
print( f"No fall {len(BASEFILENAMES['no_fall'])}")


Fall 25
No fall 23


In [16]:
make_composite_images(BASEFILENAMES['fall'],  final_width=800, final_height=800, save_dir='../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali')

✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali/01_1.jpg (rotation -30°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali/01_2.jpg (rotation -15°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali/01_3.jpg (rotation 0°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali/01_4.jpg (rotation 15°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali/01_5.jpg (rotation 30°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali/02_1.jpg (rotation -30°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/Fall/Square_Image_Rotated_Vali/02_2.jpg (rotation -15°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos -

In [17]:
make_composite_images(BASEFILENAMES['no_fall'],  final_width=800, final_height=800, save_dir='../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali')

✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali/01_1.jpg (rotation -30°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali/01_2.jpg (rotation -15°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali/01_3.jpg (rotation 0°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali/01_4.jpg (rotation 15°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali/01_5.jpg (rotation 30°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali/02_1.jpg (rotation -30°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-Detection-in-Videos - Validation/No_Fall/Square_Image_Rotated_Vali/02_2.jpg (rotation -15°)
✅ Saved ../GMDCSA24-A-Dataset-for-Human-Fall-